# 8. Risk parity and covariance shrinkage

**Level:** L2. **Tags:** `portfolio`, `risk-parity`, `covariance`.
**Library:** `qp_bt.portfolio` (`python/backtest`).

**Plain idea.** Equal weights let the most volatile asset dominate the risk.
Risk parity picks the weights so that every asset adds the same amount to
portfolio variance. A sample covariance from a short history is noisy, so it
is shrunk toward a simple target first.

**Formal statement.** Risk contribution $RC_i = w_i(\Sigma w)_i$, with
$\sum_i RC_i = w^\top\Sigma w$; risk parity solves $RC_i = RC_j$ for all
$i, j$ with $w \ge 0$ and $\sum w = 1$ (Maillard, Roncalli and Teiletche,
*Journal of Portfolio Management*, 2010; solved by Newton's method as in
Spinu, 2013). Ledoit and Wolf (*Journal of Multivariate Analysis* 88, 2004)
shrink the sample covariance $S$ to $\delta\mu I + (1-\delta)S$ with
$\mu = \mathrm{tr}(S)/N$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import numpy as np
from qp_bt.portfolio.covariance import ledoit_wolf
from qp_bt.portfolio.optimise import risk_contributions, risk_parity

## Task 1: risk-parity weights for three assets

Vols 10%, 20% and 30%, correlation 0.3 between every pair. Build `cov` and
store the weights as `w`.

In [ ]:
vol = np.array([0.10, 0.20, 0.30])
corr = np.full((3, 3), 0.3) + 0.7 * np.eye(3)
cov = np.outer(vol, vol) * corr
w = risk_parity(cov)
print("weights", np.round(w, 6), " risk contributions", np.round(risk_contributions(w, cov), 8))

**Independent check.** With one common correlation the equal-risk weights
are proportional to $1/\sigma_i$ (Maillard et al. 2010, section 2), which
needs no solver.

In [ ]:
inv = 1 / vol
assert np.allclose(w, inv / inv.sum(), atol=1e-10)
rc = w * (cov @ w)
assert np.allclose(rc, rc.mean(), atol=1e-12)

## Task 2: shrink a noisy covariance

Draw 60 days of returns for 20 assets from a seeded normal with identity
covariance (`np.random.default_rng(7)`). Store the shrunk matrix and its
intensity as `sigma_lw, delta`.

In [ ]:
x = np.random.default_rng(7).standard_normal((60, 20))
sigma_lw, delta = ledoit_wolf(x)
print(f"shrinkage intensity {delta:.4f}")

**Independent check.** The definition $\delta\mu I + (1-\delta)S$ rebuilt by
hand, and the shrunk matrix is closer to the true identity than $S$ is.

In [ ]:
xc = x - x.mean(axis=0)
s = xc.T @ xc / len(x)
mu = np.trace(s) / 20
assert 0 < delta < 1
assert np.allclose(sigma_lw, delta * mu * np.eye(20) + (1 - delta) * s, atol=1e-14)
assert np.linalg.norm(sigma_lw - np.eye(20)) < np.linalg.norm(s - np.eye(20))

**Takeaways.** Risk parity needs only a covariance, so the quality of that
covariance decides the result; shrinkage is the cheap first fix.